# Experiment: is a single-file copy of a block on Google Drive faster than downloading it again? (CPU)

The development block was downloaded again in every session, and most of that time is unpacking the compressed LiDAR
layer. This test packs the unpacked block into ONE uncompressed file on Drive, then measures how long it takes to
bring it back.

**Outcome: not adopted.** The copy came back faster in this one test, but it was read in the same session that wrote
it, and the download it was compared with was unusually slow. Decompressing on all cores
(`parallel_unpack_speed_test`) removed the problem at its source instead.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
SPLIT, BLOCK = "val", 11
LAYERS = ["camera_keyframes", "lidar_motion_compensated_keyframes"]

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Start the session ---
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, require_gpu=False)

GPU: none (fine for download and inspection)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)


In [4]:
# --- 4. Baseline: the normal download and unpack, timed ---
import shutil, subprocess, time
from vggt_aura import aura_data as ad, pipeline as pl

chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
scene_ids = ad.block_scene_ids(scene_blocks, SPLIT, BLOCK)
source = session.data_root / pl.block_tag(SPLIT, BLOCK)
shutil.rmtree(source, ignore_errors=True)
started = time.time()
ad.download_block(source, SPLIT, BLOCK, scene_ids, LAYERS)
DOWNLOAD_S = time.time() - started
size_gb = sum(f.stat().st_size for f in source.rglob("*") if f.is_file()) / 1e9
files = sum(1 for f in source.rglob("*") if f.is_file())
print(f"normal route: {DOWNLOAD_S / 60:.1f} min | {size_gb:.1f} GB unpacked | {files} files")

$ /usr/bin/python3 -m fzi_aura.download /content/data/fzi-aura/val_block000011 --revision 3404bd6b8fcd6eed53a0ec7610650a6393aabb49 --splits val --scene-ids-file /content/data/fzi-aura/val_block000011/_scene_ids_val_block000011.txt --layers camera_keyframes,lidar_motion_compensated_keyframes --jobs 8 --verify
normal route: 26.1 min | 13.0 GB unpacked | 16302 files


In [5]:
# --- 5. Pack into one uncompressed file on Drive (paid once) ---
cache = session.persist_root / "block_cache" / f"{pl.block_tag(SPLIT, BLOCK)}.tar"
cache.parent.mkdir(parents=True, exist_ok=True)
started = time.time()
subprocess.run(["tar", "-cf", str(cache), "-C", str(source.parent), source.name], check=True)
PACK_S = time.time() - started
print(f"packed to Drive in {PACK_S / 60:.1f} min | {cache.stat().st_size / 1e9:.1f} GB")

packed to Drive in 12.5 min | 13.1 GB


In [6]:
# --- 6. The route a later session would take: read that file back and unpack it ---
restored = session.data_root / "_restored"
shutil.rmtree(restored, ignore_errors=True)
restored.mkdir(parents=True)
started = time.time()
subprocess.run(["tar", "-xf", str(cache), "-C", str(restored)], check=True)
RESTORE_S = time.time() - started
ok = ad.block_on_disk(restored / source.name, scene_ids, LAYERS)
print(f"restored from Drive in {RESTORE_S / 60:.1f} min | usable by the toolkit: {ok}")
print()
print(f"normal download {DOWNLOAD_S / 60:.1f} min  vs  Drive cache {RESTORE_S / 60:.1f} min  ->  "
      f"{'CACHE WINS by %.1fx' % (DOWNLOAD_S / RESTORE_S) if RESTORE_S < 0.7 * DOWNLOAD_S else 'not worth it: keep downloading'}")
print("Note: the first read of a new Drive file can be slower than later ones. Re-run this cell once to see.")
shutil.rmtree(restored, ignore_errors=True)

restored from Drive in 4.5 min | usable by the toolkit: True

normal download 26.1 min  vs  Drive cache 4.5 min  ->  CACHE WINS by 5.8x
Note: the first read of a new Drive file can be slower than later ones. Re-run this cell once to see.
